# 2016~2024년 계획예산·합계출산율 지역연도 기초패널 생성

- **이슈**: #52
- **목적**: 17개 시도 × 2016~2024년의 시행계획상 당해예산과 합계출산율을 1:1 결합한다.
- **예산 기준**: 각 연도 문서의 `당해예산`만 사용한다. 다음 연도 문서의 `전년도예산`은 종료·변경 사업이 빠질 수 있어 지역 총액에 사용하지 않는다.
- **산출물**: 기초패널, 재정반응성 파생변수 패널, 전국 합계출산율 추세, QA 요약

> 이 패널의 예산은 결산액·실제 집행액이 아니라 시행계획상 당해 계획예산이다.

## 1. 공통 설정

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(42)

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.features.analysis_panel import (  # noqa: E402
    add_fiscal_response_features,
    build_budget_fertility_panel,
    load_budget_qa_panel,
    load_current_budget_panel,
    load_fertility_panel,
    validate_budget_totals_against_sources,
)

YEARS = list(range(2016, 2025))
INTERIM_DIR = repo_root / "data" / "interim"
REPORTS_DIR = repo_root / "reports"
MAPPING_PATH = repo_root / "data" / "lookup" / "시도_지역코드_매핑.csv"
FERTILITY_PATH = (
    repo_root / "data" / "raw" / "출산동향" / "2016-2025_시도별_출생아수_합계출산율_20260703.csv"
)
OUTPUT_DIR = repo_root / "data" / "processed" / "analysis"

region_mapping = pd.read_csv(MAPPING_PATH)
REGIONS = region_mapping["지역"].tolist()
len(REGIONS), REGIONS

(17,
 ['서울',
  '부산',
  '대구',
  '인천',
  '광주',
  '대전',
  '울산',
  '세종',
  '경기',
  '강원',
  '충북',
  '충남',
  '전북',
  '전남',
  '경북',
  '경남',
  '제주'])

## 2. 입력 파일 완전성 확인

In [2]:
input_inventory = pd.DataFrame(
    {
        "연도": YEARS,
        "시도별_long_파일수": [
            len(list(INTERIM_DIR.glob(f"*/{year}_*_세부사업_정제_long.csv"))) for year in YEARS
        ],
        "QA_파일존재": [
            (REPORTS_DIR / "yearly" / str(year) / f"{year}_전국_QA_검증결과.csv").exists()
            for year in YEARS
        ],
    }
)

assert input_inventory["시도별_long_파일수"].eq(17).all()
assert input_inventory["QA_파일존재"].all()
assert FERTILITY_PATH.exists()
display(input_inventory)

,연도,시도별_long_파일수,QA_파일존재
0,2016,17,True
1,2017,17,True
2,2018,17,True
3,2019,17,True
4,2020,17,True
5,2021,17,True
6,2022,17,True
7,2023,17,True
8,2024,17,True


## 3. 당해 계획예산 패널 생성

세부사업 예산 결측은 0으로 추정하지 않고 합계에서 제외한 뒤 `예산결측_사업수`로 기록한다. 음수 예산도 임의 수정하지 않고 `음수예산_사업수`로 표시한다.

In [3]:
budget_panel = load_current_budget_panel(
    INTERIM_DIR,
    expected_regions=REGIONS,
    expected_years=YEARS,
)

print("예산 패널 행 수:", len(budget_panel))
print("지역 수:", budget_panel["지역"].nunique())
print("연도 수:", budget_panel["연도"].nunique())
display(budget_panel.head())

예산 패널 행 수: 153
지역 수: 17
연도 수: 9


,지역,연도,당해계획예산_백만원,세부사업수,예산금액_존재_사업수,예산결측_사업수,음수예산_사업수
0,강원,2016,966330.0,251,250,1,0
1,강원,2017,1189656.0,276,276,0,0
2,강원,2018,1070709.0,77,77,0,0
3,강원,2019,1610182.0,428,428,0,0
4,강원,2020,1479594.0,470,470,0,0


In [4]:
budget_quality_summary = pd.DataFrame(
    {
        "항목": [
            "지역연도 조합",
            "세부사업 예산 결측 건수",
            "음수 예산 건수",
            "지역연도 합계 결측 건수",
        ],
        "값": [
            len(budget_panel),
            int(budget_panel["예산결측_사업수"].sum()),
            int(budget_panel["음수예산_사업수"].sum()),
            int(budget_panel["당해계획예산_백만원"].isna().sum()),
        ],
    }
)
display(budget_quality_summary)

display(
    budget_panel.loc[budget_panel["예산결측_사업수"].gt(0) | budget_panel["음수예산_사업수"].gt(0)]
)

,항목,값
0,지역연도 조합,153
1,세부사업 예산 결측 건수,287
2,음수 예산 건수,1
3,지역연도 합계 결측 건수,0


,지역,연도,당해계획예산_백만원,세부사업수,예산금액_존재_사업수,예산결측_사업수,음수예산_사업수
0,강원,2016,966330.00,251,250,1,0
9,경기,2016,4814583.00,126,124,2,0
10,경기,2017,5514545.00,123,118,5,0
11,경기,2018,7061650.00,943,879,64,0
13,경기,2020,10166623.00,1010,1003,7,0
17,경기,2024,13821287.00,1315,1302,13,0
18,경남,2016,1481184.00,319,318,1,0
19,경남,2017,1707499.35,326,323,3,0
22,경남,2020,1717208.36,657,651,6,0
27,경북,2016,1659180.00,417,416,1,0


## 4. 합계출산율 패널 변환

In [5]:
fertility_panel, nationwide_fertility = load_fertility_panel(
    FERTILITY_PATH,
    MAPPING_PATH,
    expected_years=YEARS,
)

print("합계출산율 패널 행 수:", len(fertility_panel))
print("합계출산율 결측 수:", fertility_panel["합계출산율"].isna().sum())
display(fertility_panel.head())
display(nationwide_fertility)

합계출산율 패널 행 수: 153
합계출산율 결측 수: 0


,지역,연도,합계출산율
0,강원,2016,1.237
1,강원,2017,1.123
2,강원,2018,1.067
3,강원,2019,1.082
4,강원,2020,1.036


,연도,합계출산율
0,2016,1.172
1,2017,1.052
2,2018,0.977
3,2019,0.918
4,2020,0.837
5,2021,0.808
6,2022,0.778
7,2023,0.721
8,2024,0.748


## 5. 연도별 예산 QA 품질정보 집계

In [6]:
qa_panel = load_budget_qa_panel(REPORTS_DIR, expected_years=YEARS)

qa_year_summary = qa_panel.groupby("연도", as_index=False)[
    [
        "예산_QA_그룹수",
        "예산_QA_일치건수",
        "예산_QA_불일치건수",
        "예산_QA_판정불가건수",
        "예산_QA_허용초과건수",
    ]
].sum()
display(qa_year_summary)

,연도,예산_QA_그룹수,예산_QA_일치건수,예산_QA_불일치건수,예산_QA_판정불가건수,예산_QA_허용초과건수
0,2016,95,51,42,2,7
1,2017,95,0,2,93,1
2,2018,92,1,2,89,1
3,2019,98,60,30,8,0
4,2020,100,67,33,0,2
5,2021,133,113,20,0,0
6,2022,134,119,15,0,0
7,2023,135,109,26,0,0
8,2024,136,114,22,0,6


## 6. 원자료 상세 누락 주의 플래그

2018년 강원·전남은 원자료 자체에 일부 사업의 세부내역이 없어 관측 가능한 세부사업 합계가 계획 규모를 과소대표할 수 있다. 금액을 추정해 채우지 않고 주의 문구만 결합한다.

In [7]:
quality_notes = pd.DataFrame(
    [
        {
            "지역": "강원",
            "연도": 2018,
            "원자료_누락주의": ("원자료에 자체사업 상세가 없어 세부사업 합계가 과소대표될 수 있음"),
        },
        {
            "지역": "전남",
            "연도": 2018,
            "원자료_누락주의": (
                "원자료에 자체사업 일부 세부내역이 없어 세부사업 합계가 과소대표될 수 있음"
            ),
        },
    ]
)
display(quality_notes)

,지역,연도,원자료_누락주의
0,강원,2018,원자료에 자체사업 상세가 없어 세부사업 합계가 과소대표될 수 있음
1,전남,2018,원자료에 자체사업 일부 세부내역이 없어 세부사업 합계가 과소대표될 수 있음


## 7. 계획예산·합계출산율 1:1 결합

In [8]:
base_panel = build_budget_fertility_panel(
    budget_panel,
    fertility_panel,
    expected_regions=REGIONS,
    expected_years=YEARS,
    qa_panel=qa_panel,
    quality_notes=quality_notes,
)

assert len(base_panel) == 17 * 9
assert base_panel[["지역", "연도"]].duplicated().sum() == 0
assert base_panel.groupby("연도")["지역"].nunique().eq(17).all()
assert base_panel["합계출산율"].notna().all()
assert base_panel["당해계획예산_백만원"].notna().all()
assert base_panel["당해계획예산_백만원"].ge(0).all()

budget_source_paths = sorted(
    path for year in YEARS for path in INTERIM_DIR.glob(f"*/{year}_*_세부사업_정제_long.csv")
)
budget_source_comparison = validate_budget_totals_against_sources(
    budget_panel,
    budget_source_paths,
)
assert len(budget_source_comparison) == 17 * 9
assert budget_source_comparison["집계차이_백만원"].eq(0).all()

region_year_counts = base_panel.groupby("지역", as_index=False).agg(
    시작연도=("연도", "min"),
    종료연도=("연도", "max"),
    연도수=("연도", "nunique"),
    행수=("연도", "size"),
)

assert len(region_year_counts) == 17
assert region_year_counts["연도수"].eq(9).all()
assert region_year_counts["행수"].eq(9).all()

display(region_year_counts)
display(
    pd.DataFrame(
        {
            "역대조_지역연도수": [len(budget_source_comparison)],
            "최대절대집계차이_백만원": [budget_source_comparison["집계차이_백만원"].abs().max()],
        }
    )
)
display(base_panel)

,지역,시작연도,종료연도,연도수,행수
0,강원,2016,2024,9,9
1,경기,2016,2024,9,9
2,경남,2016,2024,9,9
3,경북,2016,2024,9,9
4,광주,2016,2024,9,9
5,대구,2016,2024,9,9
6,대전,2016,2024,9,9
7,부산,2016,2024,9,9
8,서울,2016,2024,9,9
9,세종,2016,2024,9,9


,역대조_지역연도수,최대절대집계차이_백만원
0,153,0.0


,지역,연도,당해계획예산_백만원,세부사업수,예산금액_존재_사업수,예산결측_사업수,음수예산_사업수,합계출산율,예산_QA_그룹수,예산_QA_일치건수,예산_QA_불일치건수,예산_QA_판정불가건수,예산_QA_허용초과건수,예산_QA_최대절대오차율,원자료_누락주의
0,강원,2016,966330.00,251,250,1,0,1.237,6,3,3,0,0,3.04,NaN
1,강원,2017,1189656.00,276,276,0,0,1.123,6,0,0,6,0,NaN,NaN
2,강원,2018,1070709.00,77,77,0,0,1.067,3,0,0,3,0,NaN,원자료에 자체사업 상세가 없어 세부사업 합계가 과소대표될 수 있음
3,강원,2019,1610182.00,428,428,0,0,1.082,6,4,2,0,0,0.02,NaN
4,강원,2020,1479594.00,470,470,0,0,1.036,6,4,2,0,0,0.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,충북,2020,1730007.00,467,467,0,0,0.983,6,4,2,0,0,0.90,NaN
149,충북,2021,1856518.64,438,438,0,0,0.949,7,5,2,0,0,0.36,NaN
150,충북,2022,1864639.70,492,492,0,0,0.871,8,8,0,0,0,0.00,NaN
151,충북,2023,1875132.00,484,484,0,0,0.886,8,6,2,0,0,0.82,NaN


## 8. 최종 QA 요약

In [9]:
panel_qa_summary = pd.DataFrame(
    {
        "검증항목": [
            "최종 행 수",
            "지역 수",
            "연도 수",
            "지역연도 중복",
            "계획예산 합계 결측",
            "합계출산율 결측",
            "예산 결측 세부사업",
            "음수 예산 세부사업",
            "원자료 누락주의 지역연도",
        ],
        "값": [
            len(base_panel),
            base_panel["지역"].nunique(),
            base_panel["연도"].nunique(),
            int(base_panel[["지역", "연도"]].duplicated().sum()),
            int(base_panel["당해계획예산_백만원"].isna().sum()),
            int(base_panel["합계출산율"].isna().sum()),
            int(base_panel["예산결측_사업수"].sum()),
            int(base_panel["음수예산_사업수"].sum()),
            int(base_panel["원자료_누락주의"].notna().sum()),
        ],
    }
)
display(panel_qa_summary)
display(base_panel[["당해계획예산_백만원", "합계출산율"]].describe())

,검증항목,값
0,최종 행 수,153
1,지역 수,17
2,연도 수,9
3,지역연도 중복,0
4,계획예산 합계 결측,0
5,합계출산율 결측,0
6,예산 결측 세부사업,287
7,음수 예산 세부사업,1
8,원자료 누락주의 지역연도,2


,당해계획예산_백만원,합계출산율
count,1.530000e+02,153.000000
mean,2.502081e+06,0.987235
std,2.332341e+06,0.221000
min,8.944500e+04,0.552000
25%,1.161639e+06,0.827000
50%,1.757737e+06,0.952000
75%,2.862745e+06,1.122000
max,1.382129e+07,1.821000


## 9. 재정반응성 선행변수 생성

`직전1년_출산율하락도`는 분석연도보다 앞선 두 시점만 사용한다. 예를 들어 2022년 값은 `2020년 TFR - 2021년 TFR`이다.

In [10]:
fiscal_response_panel = add_fiscal_response_features(base_panel)

display(
    fiscal_response_panel[
        [
            "지역",
            "연도",
            "합계출산율",
            "전년도_합계출산율",
            "전전년도_합계출산율",
            "직전1년_출산율하락도",
            "당해계획예산_백만원",
        ]
    ].head(20)
)

,지역,연도,합계출산율,전년도_합계출산율,전전년도_합계출산율,직전1년_출산율하락도,당해계획예산_백만원
0,강원,2016,1.237,NaN,NaN,NaN,966330.00
1,강원,2017,1.123,1.237,NaN,NaN,1189656.00
2,강원,2018,1.067,1.123,1.237,0.114,1070709.00
3,강원,2019,1.082,1.067,1.123,0.056,1610182.00
4,강원,2020,1.036,1.082,1.067,-0.015,1479594.00
5,강원,2021,0.979,1.036,1.082,0.046,1968341.00
6,강원,2022,0.968,0.979,1.036,0.057,1320745.80
7,강원,2023,0.893,0.968,0.979,0.011,2579093.00
8,강원,2024,0.889,0.893,0.968,0.075,2540007.00
9,경기,2016,1.194,NaN,NaN,NaN,4814583.00


## 10. 산출물 저장

In [11]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_output = OUTPUT_DIR / "2016-2024_시도별_계획예산_합계출산율_기초패널.csv"
response_output = OUTPUT_DIR / "2016-2024_재정반응성_기초분석패널.csv"
nationwide_output = OUTPUT_DIR / "2016-2024_전국_합계출산율_추세.csv"
qa_output = OUTPUT_DIR / "2016-2024_기초패널_QA_요약.csv"

base_panel.to_csv(base_output, index=False, encoding="utf-8-sig")
fiscal_response_panel.to_csv(response_output, index=False, encoding="utf-8-sig")
nationwide_fertility.to_csv(nationwide_output, index=False, encoding="utf-8-sig")
panel_qa_summary.to_csv(qa_output, index=False, encoding="utf-8-sig")

print(base_output.relative_to(repo_root))
print(response_output.relative_to(repo_root))
print(nationwide_output.relative_to(repo_root))
print(qa_output.relative_to(repo_root))

data/processed/analysis/2016-2024_시도별_계획예산_합계출산율_기초패널.csv
data/processed/analysis/2016-2024_재정반응성_기초분석패널.csv
data/processed/analysis/2016-2024_전국_합계출산율_추세.csv
data/processed/analysis/2016-2024_기초패널_QA_요약.csv
